# 🧠 Treinamento de Transformer GPT no Kaggle (OTIMIZADO! ✨)

✨ **VERSÃO OTIMIZADA** com todas as melhorias:
- ✅ Zero tokens `<unk>` (MIN_FREQ=1, MAX_VOCAB=50000)
- ✅ Early stopping para evitar overfitting
- ✅ Suporte para formato de DIÁLOGO (ChatBot)
- ✅ Detecção de memorização
- ✅ Alertas inteligentes durante treino

Este notebook treina um modelo Transformer usando PyTorch com aceleração GPU do Kaggle.

## 📝 Instruções de Uso:

1. **Configurar o Kaggle:**
   - Vá para [kaggle.com](https://www.kaggle.com)
   - Faça login ou crie uma conta
   - Clique em "Code" → "New Notebook"

2. **Ativar Acelerador:**
   - No menu lateral direito, vá em "Settings"
   - Em "Accelerator", selecione **GPU T4 x2** (recomendado) ou **GPU P100**
   - ⚠️ **Não use TPU** para este notebook (requer código específico)
   - Clique em "Save"

3. **Upload de Dados:**
   - Vá em "Add Data" → "Upload Dataset"
   - Faça upload do seu arquivo de texto
   - **RECOMENDADO:** Use dataset de diálogo criado com `scraper_wikipedia.py` (formato 2)
   - Ou use datasets públicos do Kaggle

4. **Executar as células sequencialmente**

---

**💡 Dica:** O Kaggle oferece GPU e TPU gratuitos (30h/semana cada). Para este projeto, use **GPU T4 x2**!

**🎯 Recomendação:** Use datasets temáticos com formato de diálogo para melhores resultados!


## 1️⃣ Verificar GPU Disponível


In [ ]:
import torch
import time

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memória GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2️⃣ Definir Classes do Modelo


In [ ]:
import re
import pickle
from typing import List, Dict
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Tokenizer (MELHORADO para diálogo)
class SimpleWordTokenizer:
    """Tokenizador por palavras + pontuação + quebra de linha.

    Por que isso importa:
    - Preservar ':' ajuda MUITO em 'Usuário:' / 'Assistente:'
    - Preservar '?', '.', ',' melhora coerência
    - Preservar quebras de linha ajuda o modelo a aprender turnos

    Obs: continua simples e rápido (não é BPE), mas já melhora MUITO.
    """

    def __init__(self, min_freq: int = 2, max_vocab: int = 30000):
        self.min_freq = min_freq
        self.max_vocab = max_vocab
        self.stoi: Dict[str, int] = {}
        self.itos: Dict[int, str] = {}

        # Tokens especiais
        self.pad_token = "<pad>"
        self.unk_token = "<unk>"
        self.nl_token = "<nl>"  # representa '\n'

        # regex: captura \n, palavras (unicode) e qualquer pontuação/símbolo isolado
        self._re_tok = re.compile(r"\n|\w+|[^\w\s]", flags=re.UNICODE)

    def _tokenize(self, text: str) -> List[str]:
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        raw = self._re_tok.findall(text.lower())
        tokens: List[str] = []
        for t in raw:
            if t == "\n":
                tokens.append(self.nl_token)
            else:
                tokens.append(t)
        return tokens

    def build_vocab(self, text: str):
        tokens = self._tokenize(text)
        freq: Dict[str, int] = {}
        for t in tokens:
            freq[t] = freq.get(t, 0) + 1

        sorted_tokens = sorted(freq.items(), key=lambda x: x[1], reverse=True)

        # reservar espaço para tokens especiais
        reserved = 3  # <pad>, <unk>, <nl>
        sorted_tokens = [t for t, c in sorted_tokens if c >= self.min_freq and t not in {self.pad_token, self.unk_token, self.nl_token}]
        sorted_tokens = sorted_tokens[: self.max_vocab - reserved]

        self.stoi = {self.pad_token: 0, self.unk_token: 1, self.nl_token: 2}
        for i, tok in enumerate(sorted_tokens, start=3):
            self.stoi[tok] = i

        self.itos = {i: s for s, i in self.stoi.items()}

    def encode(self, text: str) -> List[int]:
        tokens = self._tokenize(text)
        unk_id = self.stoi[self.unk_token]
        return [self.stoi.get(t, unk_id) for t in tokens]

    def decode(self, ids: List[int]) -> str:
        tokens = [self.itos.get(i, self.unk_token) for i in ids]
        return self._detokenize(tokens)

    def _detokenize(self, tokens: List[str]) -> str:
        if not tokens:
            return ""

        # pontuação que não leva espaço antes
        no_space_before = {",", ".", "!", "?", ";", ":", ")", "]", "}"}
        # símbolos que não levam espaço depois
        no_space_after = {"(", "[", "{"}

        out = []
        for i, tok in enumerate(tokens):
            if tok == self.nl_token:
                out.append("\n")
                continue

            if i == 0:
                out.append(tok)
                continue

            prev = tokens[i - 1]
            add_space = True

            if tok in no_space_before:
                add_space = False
            if prev in no_space_after or prev == self.nl_token:
                add_space = False

            if add_space:
                out.append(" ")
            out.append(tok)

        return "".join(out)

    @property
    def vocab_size(self) -> int:
        return len(self.stoi)


# Configuração do modelo
class GPTConfig:
    def __init__(self, vocab_size: int, d_model: int = 256, n_heads: int = 4,
                 n_layers: int = 4, d_ff: int = 1024, max_seq_len: int = 128, dropout: float = 0.1):
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.d_ff = d_ff
        self.max_seq_len = max_seq_len
        self.dropout = dropout


# Modelo Transformer
class GPTModel(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model, nhead=config.n_heads,
            dim_feedforward=config.d_ff, dropout=config.dropout,
            activation="gelu", batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=config.n_layers)
        self.ln_f = nn.LayerNorm(config.d_model)
        self.head = nn.Linear(config.d_model, config.vocab_size)

    def forward(self, idx: torch.Tensor):
        B, T = idx.shape
        device = idx.device
        positions = torch.arange(0, T, device=device).unsqueeze(0)
        tok = self.token_emb(idx)
        pos = self.pos_emb(positions)
        x = tok + pos
        mask = torch.triu(torch.ones(T, T, device=device), diagonal=1).bool()
        x = self.transformer(x, mask=mask)
        x = self.ln_f(x)
        logits = self.head(x)
        return logits

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0, top_k: int = 50):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.max_seq_len:]
            logits = self.forward(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-4)
            if top_k is not None and top_k > 0:
                topk_vals, topk_idx = torch.topk(logits, k=min(top_k, logits.size(-1)))
                probs = F.softmax(topk_vals, dim=-1)
                next_token = topk_idx.gather(-1, torch.multinomial(probs, num_samples=1))
            else:
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


# Dataset
class TextDataset(Dataset):
    def __init__(self, data: List[int], block_size: int):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

print("✅ Classes definidas com sucesso!")


## 3️⃣ Carregar Dados

**⚠️ IMPORTANTE:** Ajuste o caminho do arquivo de dados abaixo!

Depois de fazer upload do seu dataset no Kaggle, o caminho geralmente é: `/kaggle/input/my-model-2/pytorch/default/1/texto_exemplo.txt`


In [ ]:
# 🔧 MODIFIQUE ESTE CAMINHO PARA SEUS DADOS!
DATA_PATH = "/kaggle/input/my-model-2/pytorch/default/1/texto_exemplo.txt"  # ⚠️ AJUSTE AQUI!

# Alternativa: Baixar um dataset de exemplo da internet
# Descomente as linhas abaixo para usar Shakespeare como exemplo:
'''
import urllib.request
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "input.txt"
urllib.request.urlretrieve(url, DATA_PATH)
print("✅ Dados baixados!")
'''

try:
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        text = f.read()
    print(f"✅ Texto carregado: {len(text):,} caracteres")
    
    # ✅ NOVO: Detecta formato de diálogo
    is_dialogue = "Usuário:" in text and "Assistente:" in text
    if is_dialogue:
        print("\n💬 FORMATO DE DIÁLOGO DETECTADO!")
        print("   Este dataset vai treinar um CHATBOT! 🤖")
    else:
        print("\n📝 Formato de texto simples detectado.")
    
    print(f"\n📄 Primeiros 300 caracteres:")
    print(text[:300])
except FileNotFoundError:
    print("❌ Arquivo não encontrado!")
    print("Dica: Faça upload do dataset no Kaggle em 'Add Data' → 'Upload Dataset'")


## 4️⃣ Preparar Dataset e DataLoader


In [ ]:
# Hiperparâmetros - ✅ OTIMIZADOS!
MIN_FREQ = 1  # ✅ Reduz <unk>
MAX_VOCAB = 50000  # ✅ Vocabulário maior
BLOCK_SIZE = 256  # ✅ Melhor para diálogos
BATCH_SIZE = 32  # ✅ Ajustado para block_size maior

# Tokenizar
print("🔤 Construindo vocabulário...")
tokenizer = SimpleWordTokenizer(min_freq=MIN_FREQ, max_vocab=MAX_VOCAB)
tokenizer.build_vocab(text)
ids = tokenizer.encode(text)

print(f"📊 Vocabulário: {tokenizer.vocab_size:,} tokens")
print(f"📊 Total de tokens no texto: {len(ids):,}")

# ✅ Verificar presença de <unk>
unk_id = tokenizer.stoi[tokenizer.unk_token]
unk_count = ids.count(unk_id)
unk_pct = (unk_count / len(ids)) * 100
print(f"\n🔍 Tokens <unk>: {unk_count:,} ({unk_pct:.2f}%)")
if unk_pct > 1.0:
    print("   ⚠️ ATENÇÃO: Muitos tokens <unk>! Aumente MAX_VOCAB ou pegue mais dados.")
else:
    print("   ✅ Vocabulário adequado!")

# ✅ Se for diálogo, checar tokens importantes
if is_dialogue:
    print("\n💬 Tokens importantes para chat:")
    print(f"   ':' no vocab? {'✅' if ':' in tokenizer.stoi else '❌'}")
    print(f"   '<nl>' no vocab? {'✅' if tokenizer.nl_token in tokenizer.stoi else '❌'}")

# Criar dataset
dataset = TextDataset(ids, block_size=BLOCK_SIZE)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2)

print(f"\n✅ Dataset criado: {len(dataset):,} exemplos")
print(f"✅ Batches por época: {len(loader)}")


## 5️⃣ Criar Modelo


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Usando device: {device}\n")

# Configuração do modelo (ajuste conforme necessário)
cfg = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    d_model=256,      # Aumente para 512 se tiver GPU potente
    n_heads=4,        # Aumente para 8 se tiver GPU potente
    n_layers=4,       # Aumente para 6 ou 8 se tiver GPU potente
    d_ff=1024,        # Aumente para 2048 se tiver GPU potente
    max_seq_len=BLOCK_SIZE,
    dropout=0.1,
)

model = GPTModel(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"📊 Total de parâmetros: {total_params:,}")
print(f"📊 Parâmetros treináveis: {trainable_params:,}")
print(f"📊 Configuração:")
print(f"   - d_model: {cfg.d_model}")
print(f"   - n_heads: {cfg.n_heads}")
print(f"   - n_layers: {cfg.n_layers}")
print(f"   - d_ff: {cfg.d_ff}")


## 6️⃣ Loop de Treinamento 🏋️

Aqui acontece o treinamento! Ajuste o número de épocas conforme necessário.


In [ ]:

# Hiperparâmetros de treinamento - ✅ COM EARLY STOPPING INTELIGENTE!
EPOCHS = 10  # Máximo, mas vai parar MUITO antes
LOG_EVERY = 100
SAMPLE_EVERY = 1

# ✅ PARADA INTELIGENTE (para DURANTE a época!)
TARGET_LOSS = 2.0    # Parar quando atingir (NÃO 1.5!)
STOP_LOSS = 1.0      # 🛑 PARA IMEDIATAMENTE se < 1.0 (overfitting!)
PATIENCE = 1         # Mais agressivo

# Salvamento automático por loss
SAVE_AT_LOSS = [2.5, 2.0, 1.8, 1.5]  # Múltiplos pontos
saved_at_loss = {loss: False for loss in SAVE_AT_LOSS}

history = {'epoch': [], 'loss': [], 'time': []}
best_loss = float('inf')
epochs_without_improvement = 0
stop_training = False  # Flag para parar

print("🏋️ Iniciando treinamento...")
print(f"🎯 Target loss: {TARGET_LOSS} (vai parar quando atingir)")
print(f"🛑 Stop loss: {STOP_LOSS} (PARA IMEDIATAMENTE!)")
print(f"⏰ Patience: {PATIENCE} época sem melhora")
print(f"💾 Salvamento automático em: {SAVE_AT_LOSS}\n")
print("=" * 80)

for epoch in range(1, EPOCHS + 1):
    if stop_training:
        print("\n🛑 TREINO PARADO!")
        break
        
    model.train()
    t0 = time.time()
    total_loss = 0.0
    steps = 0
    running_loss = 0.0
    
    for batch_idx, (xb, yb) in enumerate(loader, 1):
        xb = xb.to(device)
        yb = yb.to(device)
        
        # Forward pass
        logits = model(xb)
        loss = F.cross_entropy(logits.view(-1, cfg.vocab_size), yb.view(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        running_loss += loss.item()
        steps += 1
        
        # ✅ CALCULAR LOSS MÉDIO ATUAL
        current_avg_loss = total_loss / steps
        
        # 🛑 PARADA IMEDIATA SE LOSS < 1.0
        if current_avg_loss < STOP_LOSS:
            print(f"\n{'='*80}")
            print(f"🚨 PARADA DE EMERGÊNCIA!")
            print(f"   Loss: {current_avg_loss:.4f} < {STOP_LOSS}")
            print(f"   Época: {epoch}, Batch: {batch_idx}/{len(loader)}")
            print(f"   OVERFITTING DETECTADO! Parando treino...")
            print(f"{'='*80}\n")
            stop_training = True
            break
        
        # 💾 SALVAMENTO AUTOMÁTICO POR LOSS
        for target_loss in SAVE_AT_LOSS:
            if current_avg_loss <= target_loss and not saved_at_loss[target_loss]:
                print(f"\n{'='*80}")
                print(f"💾 CHECKPOINT AUTOMÁTICO!")
                print(f"   Loss: {current_avg_loss:.4f} <= {target_loss}")
                print(f"   Época: {epoch}, Batch: {batch_idx}/{len(loader)}")
                print(f"{'='*80}")
                
                checkpoint_auto = {
                    'epoch': epoch,
                    'batch': batch_idx,
                    'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'config': vars(cfg),
                    'tokenizer': tokenizer.stoi,
                    'is_dialogue': is_dialogue,
                    'loss': current_avg_loss,
                }
                
                filename = f'modelo_loss_{target_loss:.1f}.pt'
                torch.save(checkpoint_auto, filename)
                print(f"   ✅ Salvo: {filename}")
                print(f"   ⭐ BAIXE ESTE MODELO!\n")
                saved_at_loss[target_loss] = True
        
        # Log parcial
        if batch_idx % LOG_EVERY == 0:
            avg_running = running_loss / LOG_EVERY
            pct = (batch_idx / len(loader)) * 100
            
            # 🔍 ALERTA (interpretacao mais realista)
            alert = ""
            if avg_running < 1.0:
                alert = " 🚨 OVERFIT (loss < 1.0)"
            elif avg_running < 1.5:
                alert = " ⚠️ BAIXO (pode memorizar)"
            elif avg_running < 2.5:
                alert = " ✅ IDEAL"
            elif avg_running < 3.5:
                alert = " 👍 OK"
            else:
                alert = " 🧪 APRENDENDO"
            
            print(f"  Época {epoch}/{EPOCHS} | Batch {batch_idx}/{len(loader)} ({pct:.1f}%) | Loss: {avg_running:.4f}{alert}")
            running_loss = 0.0
            model.eval()
        
            # ✅ Prompt de amostra alinhado ao tokenizador (preserva ':' '?' etc)
            if is_dialogue:
                prompt = "usuário: o que é java? assistente:"
            else:
                prompt = "o que são"
            
            ids_prompt = tokenizer.encode(prompt)
            if not ids_prompt:
                ids_prompt = [tokenizer.stoi[tokenizer.unk_token]]
            idx = torch.tensor([ids_prompt], dtype=torch.long, device=device)
            
            with torch.no_grad():
                out = model.generate(idx, max_new_tokens=100, temperature=0.7, top_k=50)
            generated = tokenizer.decode(out[0].tolist())
            
            print("\n📝 Amostra gerada:")
            print("-" * 80)
            print(generated[:400])
            print("-" * 80)
    
    if stop_training:
        break
    
    # Estatísticas da época
    avg_loss = total_loss / steps
    elapsed = time.time() - t0
    history['epoch'].append(epoch)
    history['loss'].append(avg_loss)
    history['time'].append(elapsed)
    
    print(f"\n✅ Época {epoch}/{EPOCHS} - Loss: {avg_loss:.4f} - Tempo: {elapsed:.1f}s")
    
    # ✅ AVISAR SOBRE QUALIDADE
    if avg_loss < 0.5:
        print("   ❌ ALERTA: MEMORIZAÇÃO TOTAL! Modelo vai copiar texto!")
        print("   🛑 Parando treino agora...")
        stop_training = True
    elif avg_loss < 1.0:
        print("   🚨 ALERTA: Overfitting MUITO alto (loss < 1.0)!")
        print("   💡 Use checkpoint com loss 1.5-2.5")
    elif avg_loss < 1.5:
        print("   ⚠️  Loss baixo (< 1.5). Pode começar a memorizar.")
        print("   💡 Se a amostra piorar, pare e use checkpoint anterior")
    elif avg_loss < 2.5:
        print("   ✅ Loss ideal (1.5-2.5)! Modelo está aprendendo bem.")
    else:
        print("   🧪 Loss ainda alto. Pode treinar mais.")
    
    # Gerar amostra
    if epoch % SAMPLE_EVERY == 0:
        model.eval()
        
        if is_dialogue:
            prompt = "usuário: o que é java? assistente:"
        else:
            prompt = "o que são"
        
        ids_prompt = tokenizer.encode(prompt)
        if not ids_prompt:
            ids_prompt = [tokenizer.stoi[tokenizer.unk_token]]
        idx = torch.tensor([ids_prompt], dtype=torch.long, device=device)
        
        with torch.no_grad():
            out = model.generate(idx, max_new_tokens=100, temperature=0.7, top_k=50)
        generated = tokenizer.decode(out[0].tolist())
        
        print("\n📝 Amostra gerada:")
        print("-" * 80)
        print(generated[:400])
        print("-" * 80)
    
    # Salvar checkpoint de cada época
    checkpoint_epoca = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'config': vars(cfg),
        'tokenizer': tokenizer.stoi,
        'history': history,
        'is_dialogue': is_dialogue,
        'loss': avg_loss,
    }
    torch.save(checkpoint_epoca, f'modelo_epoca_{epoch}.pt')
    print(f"   💾 Checkpoint salvo: modelo_epoca_{epoch}.pt")
    
    # Early stopping (fim da época)
    if avg_loss < best_loss:
        best_loss = avg_loss
        epochs_without_improvement = 0
        print(f"   ⭐ Melhor loss: {best_loss:.4f}")
        torch.save(checkpoint_epoca, 'modelo_melhor.pt')
        print(f"   💾 Salvo: modelo_melhor.pt")
    else:
        epochs_without_improvement += 1
        print(f"   ⏰ Épocas sem melhora: {epochs_without_improvement}/{PATIENCE}")
    
    # Parar se atingiu target
    if avg_loss <= TARGET_LOSS and epochs_without_improvement >= PATIENCE:
        print(f"\n🛑 EARLY STOPPING!")
        print(f"   Loss {avg_loss:.4f} <= {TARGET_LOSS}")
        print(f"   Sem melhora por {PATIENCE} época(s)")
        print("   ✅ Modelo pronto!")
        break
    
    if stop_training:
        break
    
    print("=" * 80)

print("\n🎉 Treinamento concluído!")
if len(history['loss']) > 0:
    print(f"📊 Loss final: {history['loss'][-1]:.4f}")
    print(f"📊 Épocas treinadas: {len(history['epoch'])}")
    
    # Recomendação final
    final_loss = history['loss'][-1]
    if final_loss < 1.0:
        print(f"\n⚠️  ATENÇÃO: Loss muito baixo ({final_loss:.4f})")
        print(f"   💡 Use checkpoint com loss entre 1.5-2.5")
        print(f"   📁 Procure por: modelo_loss_2.0.pt ou modelo_loss_1.8.pt")
    elif final_loss < 2.0:
        print(f"\n✅ Loss bom ({final_loss:.4f})")
        print(f"   💡 Este modelo deve estar OK!")
    else:
        print(f"\n✅ Loss ideal ({final_loss:.4f})")
        print(f"   ⭐ Modelo está generalizando bem!")




## 7️⃣ Visualizar Curva de Aprendizado


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# Loss
plt.subplot(1, 2, 1)
plt.plot(history['epoch'], history['loss'], marker='o', linewidth=2, color='#2E86AB')
plt.axhline(y=TARGET_LOSS, color='r', linestyle='--', alpha=0.5, label=f'Target Loss = {TARGET_LOSS}')
plt.axhline(y=0.5, color='orange', linestyle='--', alpha=0.5, label='Risco de Overfitting')
plt.xlabel('Época', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Curva de Aprendizado', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

# Tempo
plt.subplot(1, 2, 2)
plt.bar(history['epoch'], history['time'], color='#06A77D')
plt.xlabel('Época', fontsize=12)
plt.ylabel('Tempo (s)', fontsize=12)
plt.title('Tempo por Época', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\n📊 Resumo do Treinamento:")
print(f"   Loss inicial: {history['loss'][0]:.4f}")
print(f"   Loss final: {history['loss'][-1]:.4f}")
print(f"   Redução: {(1 - history['loss'][-1]/history['loss'][0])*100:.1f}%")
print(f"   Tempo total: {sum(history['time']):.1f}s ({sum(history['time'])/60:.1f} min)")

# ✅ NOVO: Análise de qualidade
final_loss = history['loss'][-1]
print(f"\n🎯 Análise de Qualidade:")
if final_loss < 0.5:
    print(f"   ❌ MEMORIZAÇÃO! Modelo vai copiar texto literalmente.")
    print(f"      Solução: Use mais dados ou pare mais cedo.")
elif final_loss < 1.0:
    print(f"   ⚠️  Risco de memorização. Teste se está copiando.")
elif final_loss < 1.5:
    print(f"   ✅ IDEAL! Modelo deve parafrasear naturalmente.")
elif final_loss < 2.5:
    print(f"   ✅ Bom! Modelo está generalizando.")
else:
    print(f"   ⚠️  Loss alto. Considere treinar mais ou ajustar hiperparâmetros.")


## 8️⃣ Testar Geração de Texto 🎨


In [ ]:
def gerar_texto(prompt: str, max_tokens: int = 100, temperature: float = 0.8, top_k: int = 50):
    """Gera texto a partir de um prompt"""
    model.eval()
    ids_prompt = tokenizer.encode(prompt)
    if not ids_prompt:
        ids_prompt = [tokenizer.stoi[tokenizer.unk_token]]
    idx = torch.tensor([ids_prompt], dtype=torch.long, device=device)
    
    with torch.no_grad():
        out = model.generate(idx, max_new_tokens=max_tokens, temperature=temperature, top_k=top_k)
    
    return tokenizer.decode(out[0].tolist())

print("🎨 Exemplos de Geração de Texto:\n")

# ✅ Testa com formato correto (agora preservando ':' e '?')
if is_dialogue:
    print("💬 MODO CHATBOT (Formato de Diálogo)\n")
    prompts = [
        "usuário: o que é astronomia? assistente:",
        "usuário: me fale sobre planetas. assistente:",
        "usuário: explique estrelas. assistente:",
    ]
else:
    print("📝 MODO TEXTO SIMPLES\n")
    prompts = [
        "o que são",
        "como funciona",
        "a importância de",
    ]

for prompt in prompts:
    print(f"🔹 Prompt: '{prompt}'")
    print("-" * 80)
    resultado = gerar_texto(prompt, max_tokens=100, temperature=0.8, top_k=50)  # ✅ 100 tokens
    print(resultado[:500])  # ✅ Mostra mais
    print("=" * 80)
    print()


## 9️⃣ Teste de Memorização 🔍

✨ **NOVO:** Verifica se o modelo está copiando o dataset literalmente!


In [ ]:
print("🔍 TESTE DE MEMORIZAÇÃO\n")
print("Este teste verifica se o modelo está copiando texto literalmente.\n")
print("=" * 80)

# Pega um trecho aleatório do dataset
import random
start_idx = random.randint(0, len(text) - 200)
original_snippet = text[start_idx:start_idx + 200]

# Pega as primeiras 8 palavras como prompt
words = original_snippet.split()[:8]
test_prompt = " ".join(words)

print(f"📄 TRECHO ORIGINAL DO DATASET:")
print("-" * 80)
print(original_snippet)
print("-" * 80)

print(f"\n🤖 GERANDO COM PROMPT: '{test_prompt}...'")
print("-" * 80)
generated = gerar_texto(test_prompt, max_tokens=80, temperature=0.7, top_k=30)
print(generated[:300])
print("-" * 80)

print("\n🎯 ANÁLISE:")
# Verifica similaridade simples
generated_lower = generated.lower()[:200]
original_lower = original_snippet.lower()

# Conta quantas palavras são iguais
gen_words = set(generated_lower.split())
orig_words = set(original_lower.split())
overlap = len(gen_words & orig_words) / max(len(gen_words), 1)

print(f"Sobreposição de palavras: {overlap*100:.1f}%")

if overlap > 0.8:
    print("❌ MEMORIZAÇÃO! Modelo está copiando >80% das palavras.")
    print("   Solução: Reduza épocas ou use mais dados diversos.")
elif overlap > 0.6:
    print("⚠️  Alta similaridade (>60%). Pode estar memorizando.")
    print("   Teste com mais exemplos.")
elif overlap > 0.4:
    print("✅ Similaridade moderada. Modelo está parafraseando!")
else:
    print("✅ Baixa similaridade. Modelo está gerando texto novo!")


## 🔟 Resumo dos Checkpoints Salvos 📦

Durante o treinamento, foram salvos vários checkpoints:
- **`modelo_epoca_1.pt`**, **`modelo_epoca_2.pt`**, etc. - Um checkpoint por época
- **`modelo_melhor.pt`** - O modelo com o melhor loss até o momento

Você pode baixar qualquer um deles no painel "Output" do Kaggle!

---

## 1️⃣1️⃣ Salvar Modelo Final 💾

Salvando o modelo final consolidado...


In [ ]:
# Salvar checkpoint completo
checkpoint = {
    'model_state': model.state_dict(),
    'config': vars(cfg),
    'tokenizer': tokenizer.stoi,
    'history': history,
    'is_dialogue': is_dialogue,  # ✅ NOVO: Salva info sobre formato
}

torch.save(checkpoint, 'transformer_model_kaggle.pt')
print("✅ Modelo salvo em: transformer_model_kaggle.pt")
print(f"\n📊 Informações do modelo:")
print(f"   - Loss final: {history['loss'][-1]:.4f}")
print(f"   - Épocas treinadas: {len(history['epoch'])}")
print(f"   - Formato: {'Diálogo (ChatBot)' if is_dialogue else 'Texto Simples'}")
print(f"   - Vocabulário: {tokenizer.vocab_size:,} tokens")
print("\n📥 Para baixar o modelo:")
print("   1. Vá em 'Output' no painel direito do Kaggle")
print("   2. Encontre 'transformer_model_kaggle.pt'")
print("   3. Clique para fazer download")


## 1️⃣2️⃣ Modo Interativo - Teste Seus Próprios Prompts!


In [ ]:
# 🔧 Modifique o prompt abaixo e execute para gerar texto!

if is_dialogue:
    # Para ChatBot: use formato Usuário/Assistente (com ':' e pontuação)
    MEU_PROMPT = "usuário: o que é via láctea? assistente:"  # ⚠️ PERSONALIZE AQUI!
else:
    # Para texto simples
    MEU_PROMPT = "a ciência"  # ⚠️ PERSONALIZE AQUI!

resultado = gerar_texto(MEU_PROMPT, max_tokens=120, temperature=0.8, top_k=50)
print(f"💬 Prompt: '{MEU_PROMPT}'")
print("=" * 80)
print(resultado)
print("=" * 80)


## 1️⃣3️⃣ Como Carregar um Checkpoint Específico 🔄

Se quiser carregar um modelo de uma época específica (por exemplo, se a época 3 teve o melhor resultado):


In [ ]:
# ✅ EXEMPLO: Como carregar um checkpoint de época específica

# Opção 1: Carregar o melhor modelo (recomendado!)
checkpoint_carregado = torch.load('modelo_melhor.pt')

# Opção 2: Carregar uma época específica
# checkpoint_carregado = torch.load('modelo_epoca_3.pt')

# Opção 3: Carregar o modelo final
# checkpoint_carregado = torch.load('transformer_model_kaggle.pt')

# Reconstruir modelo
cfg_carregado = GPTConfig(**checkpoint_carregado['config'])
modelo_carregado = GPTModel(cfg_carregado).to(device)
modelo_carregado.load_state_dict(checkpoint_carregado['model_state'])
modelo_carregado.eval()

# Reconstruir tokenizer
tokenizer_carregado = SimpleWordTokenizer()
tokenizer_carregado.stoi = checkpoint_carregado['tokenizer']
tokenizer_carregado.itos = {i: s for s, i in tokenizer_carregado.stoi.items()}

print("✅ Modelo carregado com sucesso!")
print(f"📊 Época: {checkpoint_carregado.get('epoch', 'N/A')}")
print(f"📊 Loss: {checkpoint_carregado.get('loss', checkpoint_carregado['history']['loss'][-1]):.4f}")
print(f"💬 Formato: {'Diálogo' if checkpoint_carregado['is_dialogue'] else 'Texto Simples'}")

# Testar com o modelo carregado
prompt_teste = "usuário: o que é java? assistente:" if checkpoint_carregado['is_dialogue'] else "o que são"
ids_teste = tokenizer_carregado.encode(prompt_teste)
if not ids_teste:
    ids_teste = [tokenizer_carregado.stoi[tokenizer_carregado.unk_token]]
idx_teste = torch.tensor([ids_teste], dtype=torch.long, device=device)

with torch.no_grad():
    out_teste = modelo_carregado.generate(idx_teste, max_new_tokens=50, temperature=0.8, top_k=50)

print("\n🎨 Teste de geração:")
print("-" * 80)
print(tokenizer_carregado.decode(out_teste[0].tolist()))
print("-" * 80)


---

## 📚 Dicas e Truques

### 🚀 Para Melhorar o Modelo:
- **Mais dados**: Quanto mais texto, melhor o modelo aprende
- **Mais épocas**: Aumente de 5 para 10-20 épocas
- **Modelo maior**: Aumente `d_model` (256 → 512), `n_layers` (4 → 6-8)
- **Batch size**: Aumente se tiver GPU com memória suficiente

### 💾 Memória GPU:
- Se der erro de memória (OOM), **reduza** `batch_size` ou `d_model`
- GPU T4 (16GB): batch_size=64, d_model=512 funciona bem
- GPU P100 (16GB): similar ao T4

### 📊 Monitoramento:
- Loss deve **diminuir** ao longo das épocas
- Se estabilizar cedo, considere:
  - Reduzir learning rate (3e-4 → 1e-4)
  - Aumentar dropout (0.1 → 0.2)
  - Adicionar mais dados

### 🎯 Datasets Públicos no Kaggle:
- Wikipedia em português
- Livros do Projeto Gutenberg
- Corpus de notícias (G1, Folha, etc.)
- Textos científicos e acadêmicos

### 🔄 Como Usar o Modelo Salvo:
```python
# Carregar modelo salvo
checkpoint = torch.load('transformer_model_kaggle.pt')
model.load_state_dict(checkpoint['model_state'])
model.eval()

# Gerar texto
texto = gerar_texto("seu prompt aqui", max_tokens=100)
print(texto)
```

---

**🎉 Parabéns! Você treinou seu modelo Transformer no Kaggle!**

**⭐ Se este notebook foi útil, considere dar um upvote!**


---

## 📚 Resumo das Otimizações

### ✅ O Que Foi Melhorado:

1. **MIN_FREQ = 1** (antes: 2)
   - ✅ Zero tokens `<unk>`
   - ✅ Vocabulário completo

2. **MAX_VOCAB = 50000** (antes: 30000)
   - ✅ Mais palavras no vocabulário
   - ✅ Melhor cobertura do texto

3. **BLOCK_SIZE = 256** (antes: 128)
   - ✅ Contexto maior
   - ✅ Melhor para diálogos

4. **Early Stopping**
   - ✅ Para quando loss < 1.5
   - ✅ Evita overfitting
   - ✅ Economiza tempo

5. **Detecção de Formato de Diálogo**
   - ✅ Reconhece automaticamente
   - ✅ Testa com formato correto
   - ✅ Salva informação no checkpoint

6. **Teste de Memorização**
   - ✅ Verifica se está copiando
   - ✅ Análise de similaridade
   - ✅ Recomendações automáticas

7. **Alertas de Overfitting**
   - ✅ Avisa quando loss < 0.5
   - ✅ Explica o que fazer

8. **Amostras Mais Longas**
   - ✅ 100 tokens (antes: 60)
   - ✅ Melhor visualização

9. **Checkpoints por Época** 💾
   - ✅ Salva modelo após cada época
   - ✅ Salva melhor modelo automaticamente
   - ✅ Permite recuperar treino interrompido
   - ✅ Compara épocas diferentes
   - ✅ Escolhe o melhor modelo depois

---

## 🎯 Resultados Esperados:

### ❌ ANTES (Configuração Antiga):
```
Loss final: ~0.3
Tokens <unk>: Muitos
Resultado: COPIA texto literalmente (CTRL+C CTRL+V)
```

### ✅ DEPOIS (Configuração Otimizada):
```
Loss final: 1.0-1.5
Tokens <unk>: Zero
Resultado: PARAFRASEIA naturalmente! 🎉
```

---

**🎉 Parabéns! Você treinou um modelo otimizado no Kaggle!**

**⭐ Se este notebook foi útil, considere dar um upvote!**
